# Phase 8 v4 FINAL — Metadata precision-first, vá chặt people/locations

Notebook này đọc `vn_history_rag_chunks.jsonl`, trích metadata theo `chunk_id`, kiểm tra bằng rule chặt và xuất:

- `vn_history_rag_chunk_metadata.jsonl`
- `vn_history_rag_chunks_enriched.jsonl`
- báo cáo coverage, review và QC

V4 kế thừa toàn bộ bộ lọc năm, triều đại, văn kiện và sự kiện của v3, đồng thời vá thêm các lỗi `people`/`locations`: chức danh đứng riêng, mảnh tên bị cắt, cơ quan, biệt danh địa điểm, mô tả hướng, mô tả mở rộng và tên dân tộc bị nhận nhầm thành địa danh. Metadata không chắc chắn sẽ bị loại thay vì cố giữ.


In [1]:

# Cell 1 — Mount Google Drive và cài thư viện

from google.colab import drive
drive.mount('/content/drive')

!pip -q install -U "transformers>=4.46" accelerate bitsandbytes sentencepiece json-repair pandas tqdm


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 8.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 136.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 141.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.2/80.2 kB 9.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 3.0.5 which is incompatible.
cudf-cu12 26.2.1 requires pandas<2.4.0,>=2.0, but

In [2]:
# Cell 2 — Imports và cấu hình chạy FULL

import os
import re
import json
import hashlib
import unicodedata
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Optional, Tuple

import pandas as pd
import torch
from tqdm.auto import tqdm
from json_repair import repair_json

# =========================
# Đường dẫn
# =========================
DRIVE_ROOT = Path('/content/drive/MyDrive/vn_history_model_backups')
INPUT_PATH = (
    DRIVE_ROOT
    / 'rag_corpus_vn_history'
    / 'processed'
    / 'vn_history_rag_chunks.jsonl'
)

OUTPUT_DIR = DRIVE_ROOT / 'rag_corpus_vn_history' / 'metadata'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METADATA_PATH = OUTPUT_DIR / 'vn_history_rag_chunk_metadata.jsonl'
ENRICHED_PATH = OUTPUT_DIR / 'vn_history_rag_chunks_enriched.jsonl'

# V4 dùng checkpoint riêng, tuyệt đối không tái sử dụng metadata v1/v2/v3.
CHECKPOINT_PATH = OUTPUT_DIR / 'vn_history_rag_chunk_metadata.v4.checkpoint.jsonl'
ERROR_PATH = OUTPUT_DIR / 'vn_history_rag_chunk_metadata.v4.errors.jsonl'
REPORT_PATH = OUTPUT_DIR / 'vn_history_rag_chunk_metadata_report.csv'
REVIEW_PATH = OUTPUT_DIR / 'vn_history_rag_chunk_metadata_manual_review.csv'
QC_PATH = OUTPUT_DIR / 'vn_history_rag_chunk_metadata_qc.json'

# =========================
# Chế độ chạy
# =========================
# "rules_only": nhanh, precision cao nhưng people/events/documents sẽ thưa.
# "hybrid": rule + Qwen; dùng cho file chính thức.
EXTRACTION_MODE = 'hybrid'

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
LOAD_IN_4BIT = True
BATCH_SIZE = 6               # OOM thì giảm 2 hoặc 1.
MAX_INPUT_CHARS = 9000
MAX_NEW_TOKENS = 260
SAVE_EVERY = 25
RESUME = True
FORCE_REBUILD = False
LIMIT = None                 # FULL CORPUS.

KEEP_RAW_LLM_OUTPUT = False
RULES_FALLBACK_ON_LLM_ERROR = True
METADATA_VERSION = 'v4.0-final-people-location-strict'
VALIDATOR_SIGNATURE = 'phase8_v4_people_location_20260731_a'
CURRENT_YEAR = datetime.now().year

print('INPUT_PATH:', INPUT_PATH)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('LIMIT:', LIMIT)
print('METADATA_VERSION:', METADATA_VERSION)
print('VALIDATOR_SIGNATURE:', VALIDATOR_SIGNATURE)
print('CHECKPOINT_PATH:', CHECKPOINT_PATH)


INPUT_PATH: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/processed/vn_history_rag_chunks.jsonl
OUTPUT_DIR: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata
LIMIT: None
METADATA_VERSION: v4.0-final-people-location-strict
VALIDATOR_SIGNATURE: phase8_v4_people_location_20260731_a
CHECKPOINT_PATH: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata/vn_history_rag_chunk_metadata.v4.checkpoint.jsonl


In [3]:

# Cell 3 — Đọc và kiểm tra corpus

def read_jsonl(path: Path) -> List[Dict[str, Any]]:
    records = []
    with Path(path).open('r', encoding='utf-8') as f:
        for line_no, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                obj = json.loads(line)
            except json.JSONDecodeError as exc:
                raise ValueError(f'JSON lỗi tại dòng {line_no}: {exc}') from exc
            records.append(obj)
    return records

if not INPUT_PATH.exists():
    raise FileNotFoundError(
        'Không tìm thấy corpus. Kiểm tra lại đường dẫn:\n'
        f'{INPUT_PATH}'
    )

chunks = read_jsonl(INPUT_PATH)
if LIMIT is not None:
    chunks = chunks[: int(LIMIT)]

required = {'chunk_id', 'title', 'text'}
for i, chunk in enumerate(chunks[:20]):
    missing = required - set(chunk)
    if missing:
        raise ValueError(f'Chunk index {i} thiếu trường: {sorted(missing)}')

chunk_ids = [str(c.get('chunk_id', '')).strip() for c in chunks]
if any(not cid for cid in chunk_ids):
    raise ValueError('Có chunk_id rỗng.')

if len(set(chunk_ids)) != len(chunk_ids):
    dup = pd.Series(chunk_ids)[pd.Series(chunk_ids).duplicated()].unique().tolist()[:20]
    raise ValueError(f'chunk_id bị trùng. Ví dụ: {dup}')

print(f'Tổng chunks sẽ xử lý: {len(chunks):,}')
print('Columns mẫu:', sorted(chunks[0].keys()))
display(pd.DataFrame(chunks[:3]))


Tổng chunks sẽ xử lý: 58,603
Columns mẫu: ['char_len', 'chunk_filter_reason', 'chunk_history_score', 'chunk_id', 'chunk_index', 'doc_filter_reason', 'filter_version', 'hf_dataset', 'history_score', 'matched_vn_history_terms', 'raw_record_index', 'section', 'source', 'source_type', 'text', 'text_hash', 'title', 'url', 'word_len']


,chunk_id,source,source_type,title,section,url,chunk_index,text,char_len,word_len,raw_record_index,hf_dataset,history_score,chunk_history_score,doc_filter_reason,chunk_filter_reason,matched_vn_history_terms,filter_version,text_hash
0,hf_wikipedia_thành_phố_hồ_chí_minh_0000_2da892...,DataStudio/Viet-wikipedia,hf_wikipedia,Thành phố Hồ Chí Minh,,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,0,"Thành phố Hồ Chí Minh (viết tắt TP.HCM), còn đ...",2967,650,6,DataStudio/Viet-wikipedia,69,50,trusted_vn_history_title,trusted_title_and_historical_chunk,"[thành phố hồ chí minh, nguyễn hữu cảnh, chúa ...",phase2_v2_vn_history_doc_and_chunk_filter_2026_07,a381cd4796ced7f3
1,hf_wikipedia_thành_phố_hồ_chí_minh_0001_ed1d52...,DataStudio/Viet-wikipedia,hf_wikipedia,Thành phố Hồ Chí Minh,,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,1,"Nam Bộ trở thành đất vô chủ, về sau đã sáp nhậ...",2954,650,6,DataStudio/Viet-wikipedia,69,52,trusted_vn_history_title,trusted_title_and_historical_chunk,"[việt nam dân chủ cộng hòa, thành phố hồ chí m...",phase2_v2_vn_history_doc_and_chunk_filter_2026_07,b343d847a187c2fe
2,hf_wikipedia_thành_phố_hồ_chí_minh_0002_4637b9...,DataStudio/Viet-wikipedia,hf_wikipedia,Thành phố Hồ Chí Minh,,https://vi.wikipedia.org/wiki/Th%C3%A0nh%20ph%...,2,Cộng hòa Miền Nam Việt Nam tiếp quản chính quy...,2912,650,6,DataStudio/Viet-wikipedia,69,52,trusted_vn_history_title,trusted_title_and_historical_chunk,"[việt nam dân chủ cộng hòa, thành phố hồ chí m...",phase2_v2_vn_history_doc_and_chunk_filter_2026_07,c5399f5fb91e3ba0


## Nguyên tắc chống nhiễu của v4

1. `chunk_id` là khóa liên kết duy nhất; corpus gốc không bị sửa.
2. Qwen chỉ đề xuất entity xuất hiện bề mặt; rule quyết định item nào được giữ.
3. Năm phải có ngữ cảnh niên đại và không được là số đo, số liệu, khoảng thời gian tương đối hay thập niên.
4. `people` chỉ nhận tên cá nhân đầy đủ; loại dân tộc, tổ chức, chức danh đứng riêng và mảnh tên bị cắt từ một tên dài hơn.
5. `locations` chỉ nhận tên địa danh; loại cơ quan, doanh nghiệp, biệt danh, mô tả hướng, mô tả mở rộng và tên dân tộc.
6. `dynasties` match có dấu theo cụm chính xác, vì vậy `nhà hộ sinh` không thể thành `Nhà Hồ`.
7. `documents` phải có tên phân biệt; `Sắc lệnh số` đứng riêng bị loại.
8. `events` phải mang dạng tên sự kiện; câu hành động như `đổi trấn làm châu` bị loại.
9. Trước khi ghi file cuối, toàn bộ metadata được sanitize lại lần cuối và QC toàn bộ corpus.


In [4]:
# Cell 4 — Rule-based metadata, chuẩn hóa và validator precision-first

STOP_TOKENS = {
    'và', 'của', 'là', 'trong', 'với', 'tại', 'do', 'được', 'các', 'những',
    'một', 'này', 'đó', 'về', 'cho', 'từ', 'đến', 'sau', 'trước',
}

# Dynasty/chính quyền phải match chính xác và GIỮ DẤU.
DYNASTY_SURFACE_PHRASES = {
    'Nhà Ngô': ['nhà ngô', 'triều ngô'],
    'Nhà Đinh': ['nhà đinh', 'triều đinh'],
    'Nhà Tiền Lê': ['nhà tiền lê', 'triều tiền lê'],
    'Nhà Lý': ['nhà lý', 'triều lý'],
    'Nhà Trần': ['nhà trần', 'triều trần'],
    'Nhà Hồ': ['nhà hồ', 'triều hồ'],
    'Nhà Lê sơ': ['nhà lê sơ', 'triều lê sơ', 'lê sơ'],
    'Nhà Hậu Lê': ['nhà hậu lê', 'triều hậu lê'],
    'Nhà Mạc': ['nhà mạc', 'triều mạc'],
    'Chúa Trịnh': ['chúa trịnh'],
    'Chúa Nguyễn': ['chúa nguyễn'],
    'Triều Tây Sơn': ['nhà tây sơn', 'triều tây sơn'],
    'Nhà Nguyễn': ['nhà nguyễn', 'triều nguyễn'],
    'Đế quốc Khmer': ['đế quốc khmer'],
    'Việt Nam Dân chủ Cộng hòa': ['việt nam dân chủ cộng hòa'],
    'Việt Nam Cộng hòa': ['việt nam cộng hòa'],
    'Cộng hòa Miền Nam Việt Nam': ['cộng hòa miền nam việt nam'],
    'Cộng hòa xã hội chủ nghĩa Việt Nam': ['cộng hòa xã hội chủ nghĩa việt nam'],
    'Đế quốc Việt Nam': ['đế quốc việt nam'],
    'Chính quyền thực dân Pháp': ['chính quyền thực dân pháp', 'thực dân pháp'],
}

FACET_PATTERNS = {
    'bối cảnh': [r'\bboi canh\b', r'\bhoan canh lich su\b'],
    'nguyên nhân': [r'\bnguyen nhan\b', r'\bly do\b'],
    'diễn biến': [r'\bdien bien\b'],
    'kết quả': [r'\bket qua\b', r'\bhe qua\b'],
    'ý nghĩa': [r'\by nghia\b'],
    'chính sách': [r'\bchinh sach\b', r'\bchu truong\b'],
}

TOPIC_PATTERNS = {
    'quân sự': [r'\bchien tranh\b', r'\bkhang chien\b', r'\bchien dich\b', r'\btran danh\b', r'\bxam luoc\b'],
    'ngoại giao': [r'\bngoai giao\b', r'\bhiep dinh\b', r'\bhoa uoc\b', r'\bdam phan\b', r'\bbang giao\b'],
    'kinh tế': [r'\bkinh te\b', r'\bnong nghiep\b', r'\bthuong nghiep\b', r'\bcong nghiep\b', r'\bthue\b'],
    'văn hóa': [r'\bvan hoa\b', r'\bvan hoc\b', r'\bnghe thuat\b'],
    'xã hội': [r'\bxa hoi\b', r'\bdan cu\b', r'\bdan so\b'],
    'giáo dục': [r'\bgiao duc\b', r'\bquoc tu giam\b', r'\bvan mieu\b'],
    'pháp luật': [r'\bphap luat\b', r'\bbo luat\b', r'\bquoc trieu hinh luat\b'],
    'tôn giáo': [r'\bton giao\b', r'\bphat giao\b', r'\bcong giao\b'],
    'khởi nghĩa': [r'\bkhoi nghia\b'],
    'kháng chiến': [r'\bkhang chien\b'],
    'cải cách': [r'\bcai cach\b'],
    'chủ quyền': [r'\bchu quyen\b'],
    'độc lập': [r'\bdoc lap\b'],
    'thống nhất': [r'\bthong nhat\b'],
    'thuộc địa': [r'\bthuoc dia\b'],
    'cách mạng': [r'\bcach mang\b'],
    'văn kiện': [r'\bhiep dinh\b', r'\bhoa uoc\b', r'\btuyen ngon\b', r'\bchieu\b', r'\bhich\b', r'\bdai cao\b'],
}

SURFACE_FIELDS = ['people', 'events', 'locations', 'documents', 'periods']

PERIOD_CUES = (
    'thoi ', 'thoi ky', 'giai doan', 'the ky', 'bac thuoc', 'phap thuoc',
    'bao cap', 'doi moi', 'khang chien', 'chien tranh', 'trieu dai',
)
PERSON_ROLE_CUES = (
    'thu tuong', 'chu tich', 'vua', 'hoang de', 'tuong', 'dai tuong',
    'chua', 'hoang hau', 'cong chua', 'thai hau', 'lanh tu',
)
EVENT_PREFIXES = (
    'khoi nghia ', 'chien dich ', 'tran ', 'chien tranh ', 'cach mang ',
    'phong trao ', 'tong tien cong ', 'dao chinh ', 'khang chien ',
    'hoi nghi ', 'bien co ', 'nan doi ', 'cuoc chien ', 'hai chien ',
)
DOCUMENT_CUES = (
    'hiep dinh', 'hoa uoc', 'chieu ', 'hich ', 'dai cao', 'tuyen ngon',
    'bo luat', 'luat ', 'sac lenh', 'nghi quyet', 'chi thi', 'thuc luc',
    'su ky', 'ban do', 'tac pham', 'duong kach menh', 'yeu sach',
)

PERSON_FORBIDDEN_PREFIXES = (
    'nguoi ', 'dan toc ', 'chinh phu', 'hoi dong', 'bo ', 'so ', 'uy ban',
    'quoc hoi', 'quan doi', 'dang ', 'mat tran', 'trieu dinh', 'nha nuoc',
    'phong ', 'vien ', 'ban ',
    # Chức danh đứng đầu không phải tên cá nhân. Precision-first: yêu cầu LLM trả tên riêng,
    # không trả cả chức danh.
    'quan ', 'chu tich ', 'thu tuong ', 'hoang de ', 'vua ', 'dai tuong ',
    'tuong ', 'duc vua ', 'duc vuong ', 'thu linh ', 'cong chua ',
    'thai hau ', 'hoang hau ', 'quoc vuong ', 'tiet do su ', 'tri chau ',
)
PERSON_FORBIDDEN_SUBSTRINGS = (
    'tri chau', 'tiet do', 'tiet do su', 'thai thu', 'thu su', 'quan do doc',
    'thu tuong chinh phu', 'hoi dong chinh phu', 'hoi dong bo truong',
    'chu tich chinh phu', 'chuc vu', 'vien quan',
)
PERSON_FORBIDDEN_EXACT = {
    'an nam', 'dai viet', 'dai nam', 'viet nam', 'nam chieu', 'giao chau',
    'phong chau', 'chinh phu', 'nguoi viet', 'nguoi hoa',
    'duc vuong', 'chu tich chinh phu', 'thu tuong chinh phu',
    'quan thac dong', 'quan thac dong tiet do',
}
PERSON_ROLE_ONLY_WORDS = {
    'quan', 'chu', 'tich', 'thu', 'tuong', 'chinh', 'phu', 'vua', 'hoang',
    'de', 'duc', 'vuong', 'dai', 'lanh', 'dao', 'thai', 'hau', 'cong',
    'chua', 'tiet', 'do', 'su', 'tri', 'chau', 'vien', 'chuc', 'thu linh',
}
VN_SURNAMES = {
    'nguyễn', 'trần', 'lê', 'phạm', 'hoàng', 'huỳnh', 'phan', 'vũ', 'võ',
    'đặng', 'bùi', 'đỗ', 'hồ', 'ngô', 'dương', 'lý', 'đinh', 'đào', 'tô',
    'trịnh', 'mai', 'chu', 'cao', 'lương', 'hà', 'lưu', 'tạ', 'thái',
    'quách', 'tôn', 'mạc', 'chế', 'nông', 'kiều', 'khúc',
}
BIOGRAPHY_CUES = (
    'ông ', 'bà ', 'vua ', 'hoàng đế', 'tướng ', 'đại tướng', 'chủ tịch',
    'thủ tướng', 'lãnh tụ', 'sinh ', 'mất ', 'qua đời', 'trị vì',
    'nhà sử học', 'nhà văn', 'nhà thơ', 'nhà cách mạng',
)

LOCATION_FORBIDDEN_PREFIXES = (
    'chua ', 'vua ', 'hoang de', 'thu tuong', 'chu tich', 'tuong ',
    'trieu ', 'nha ', 'phong ', 'bo ', 'hoi dong', 'chinh phu', 'uy ban',
    'xi nghiep', 'cong ty', 'nha may', 'truong ', 'benh vien', 'quan khu',
    'co quan', 'don vi', 'ban ', 'so ', 'vien ', 'thu vien ', 'hoc vien ',
    'khu vuc', 'khu do thi', 'hon ngoc',
    # Mô tả phương hướng/vị trí không phải tên riêng địa danh.
    'cuc bac ', 'cuc nam ', 'cuc dong ', 'cuc tay ',
    'phia bac ', 'phia nam ', 'phia dong ', 'phia tay ',
)
LOCATION_FORBIDDEN_SUBSTRINGS = (
    'lien hiep duong sat', 'phong trung uong', 'hoi dong', 'chinh phu',
    'vien khoa hoc', 'nho o vien dong', 'cuc bac cua', 'cuc nam cua',
    'cuc dong cua', 'cuc tay cua',
)
LOCATION_FORBIDDEN_EXACT = {
    'kinh su', 'di', 'noi day', 'khu vuc nay', 'vung nay',
    'paris nho o vien dong', 'sai gon mo rong',
}
LOCATION_FORBIDDEN_SUFFIXES = (
    ' mo rong', ' thu hep', ' hien nay', ' ngay nay', ' moi', ' cu',
)
LOCATION_INTERNAL_CONNECTORS = {
    'ở', 'của', 'thuộc', 'nằm', 'gần', 'giữa',
}
LOCATION_GEO_CUES = (
    'tinh ', 'thanh pho ', 'huyen ', 'quan ', 'xa ', 'phuong ', 'thi xa ',
    'thi tran ', 'chau ', 'phu ', 'tran ', 'lang ', 'song ', 'nui ', 'dao ',
    'vinh ', 'cua khau ', 'dong bang ', 'cao nguyen ', 'mien ', 'vung ',
    'tai ', 'den ', 'tu ', 'o ',
)
LOCATION_ETHNIC_CUES = (
    'nguoi ', 'dan toc ', 'toc nguoi ', 'nhom nguoi ', 'sac toc ',
)
LOCATION_SINGLE_TOKEN_ALLOWLIST = {
    'huế', 'lào', 'xiêm', 'pháp', 'nhật', 'hàn', 'chăm',
}

GENERIC_DOCUMENTS = {
    'hiep dinh', 'hoa uoc', 'chieu', 'hich', 'dai cao', 'tuyen ngon',
    'bo luat', 'luat', 'sac lenh', 'sac lenh so', 'nghi quyet', 'chi thi',
    'thuc luc', 'su ky', 'ban do', 'tac pham', 'yeu sach',
}


def clean_text(value: Any) -> str:
    value = '' if value is None else str(value)
    return re.sub(r'\s+', ' ', value).strip()


def normalize_keep_diacritics(text: Any) -> str:
    text = unicodedata.normalize('NFC', clean_text(text).lower())
    return re.sub(r'\s+', ' ', text).strip()


def surface_text(text: Any) -> str:
    text = normalize_keep_diacritics(text)
    text = re.sub(r'[^\w\s]+', ' ', text, flags=re.UNICODE)
    return re.sub(r'\s+', ' ', text).strip()


def strip_accents(text: str) -> str:
    text = unicodedata.normalize('NFD', clean_text(text).lower())
    text = ''.join(ch for ch in text if unicodedata.category(ch) != 'Mn')
    return text.replace('đ', 'd')


def norm_surface(text: Any) -> str:
    text = strip_accents(clean_text(text))
    text = re.sub(r'[^a-z0-9]+', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def stable_unique(items: List[Any]) -> List[str]:
    seen = set()
    out = []
    for item in items or []:
        item = clean_text(item).strip(' ,;:.–—-')
        key = norm_surface(item)
        if not item or not key or key in seen:
            continue
        seen.add(key)
        out.append(item)
    return out


def contains_surface_phrase(text: Any, phrase: Any) -> bool:
    haystack = f' {surface_text(text)} '
    needle = f' {surface_text(phrase)} '
    return bool(surface_text(phrase)) and needle in haystack


def extract_rule_labels(text: str, pattern_map: Dict[str, List[str]], max_items: int) -> List[str]:
    haystack = norm_surface(text)
    out = []
    for label, patterns in pattern_map.items():
        if any(re.search(pattern, haystack, flags=re.IGNORECASE) for pattern in patterns):
            out.append(label)
            if len(out) >= max_items:
                break
    return out


def extract_dynasties(text: str, max_items: int = 8) -> List[str]:
    out = []
    for label, phrases in DYNASTY_SURFACE_PHRASES.items():
        if any(contains_surface_phrase(text, phrase) for phrase in phrases):
            out.append(label)
            if len(out) >= max_items:
                break
    return out


# -------------------------
# Năm: chỉ nhận niên đại, loại số đo/thập niên/khoảng thời gian tương đối
# -------------------------
YEAR_CONTEXT_CUES = (
    'khởi nghĩa', 'chiến dịch', 'trận', 'chiến tranh', 'cách mạng',
    'hiệp định', 'hòa ước', 'thành lập', 'ra đời', 'đổi tên', 'đăng quang',
    'trị vì', 'sinh', 'mất', 'qua đời', 'xâm lược', 'giành độc lập',
    'thống nhất', 'ban hành', 'ký kết', 'chiếm', 'thất thủ', 'tiếp quản',
)
MEASURE_UNITS_PATTERN = (
    r'(?:%|km(?:²|2|³|3)?|m(?:²|2|³|3)?|ha|người|triệu|tỷ|lần|tuổi|'
    r'tháng|ngày|độ|°c|°f|mhz|khz|hz|usd|đồng|tấn|kg|cm|mm|px|lít|lit)'
)


def valid_year_value(value: Any) -> bool:
    try:
        year = int(value)
    except (TypeError, ValueError):
        return False
    return 1 <= year <= CURRENT_YEAR


def _is_grouped_number(text: str, start: int, end: int) -> bool:
    right_group = end + 1 < len(text) and text[end] in '.,' and text[end + 1].isdigit()
    left_group = start >= 2 and text[start - 1] in '.,' and text[start - 2].isdigit()
    return right_group or left_group


def _is_decade_reference(body: str, start: int) -> bool:
    before = body[max(0, start - 40):start]
    return bool(re.search(
        r'(?:những|các|đầu|giữa|cuối)?\s*(?:những\s+)?năm\s*$',
        before,
    )) and not bool(re.search(r'(?<!những\s)(?<!các\s)\bnăm\s*$', before))


def _is_measure_or_relative_duration(body: str, start: int, end: int) -> bool:
    before = body[max(0, start - 35):start]
    after = body[end:min(len(body), end + 45)]
    around = body[max(0, start - 35):min(len(body), end + 55)]

    if re.match(r'\s*(?:[-–—]\s*\d{1,4}\s*)?%', after):
        return True
    if re.match(rf'\s*(?:[-–—]\s*\d{{1,4}}\s*)?(?<!\w){MEASURE_UNITS_PATTERN}(?!\w)', after):
        return True
    if re.search(rf'(?<!\w){MEASURE_UNITS_PATTERN}(?!\w)\s*$', before):
        return True
    if re.search(r'\b(?:triệu|tỷ)\s+năm\b', around):
        return True
    if re.search(r'\b(?:cách đây|khoảng|hơn|gần)\b.{0,25}\d{1,4}(?:\s*[-–—]\s*\d{1,4})?\s+năm\b', around):
        return True
    if re.search(r'\d{1,4}(?:\s*[-–—]\s*\d{1,4})?\s+năm\s+(?:trước|qua|trở lại|tuổi)\b', around):
        return True
    return False


def _year_match_supported(year: int, body: str, match: re.Match) -> bool:
    if not valid_year_value(year):
        return False
    if _is_grouped_number(body, match.start(), match.end()):
        return False
    if _is_measure_or_relative_duration(body, match.start(), match.end()):
        return False
    if _is_decade_reference(body, match.start()):
        return False

    before = body[max(0, match.start() - 45):match.start()]
    after = body[match.end():min(len(body), match.end() + 45)]
    context = body[max(0, match.start() - 90):min(len(body), match.end() + 90)]

    explicit_year = bool(re.search(r'\bnăm\s*$', before))
    if explicit_year:
        return True

    if re.search(r'\b(?:trước|sau)\s+công\s+nguyên\b', after):
        return True

    # Năm đơn nằm trong ngoặc, ví dụ niên hiệu (...) (854).
    immediate_before = body[max(0, match.start() - 3):match.start()]
    immediate_after = body[match.end():min(len(body), match.end() + 3)]
    if year >= 100 and '(' in immediate_before and ')' in immediate_after:
        return True

    # Khoảng niên đại chỉ nhận khi nằm trong ngoặc hoặc có chỉ báo thời gian trực tiếp.
    range_near = bool(
        re.search(r'[-–—]\s*$', before)
        or re.match(r'^\s*[-–—]\s*\d{1,4}', after)
    )
    if range_near:
        parenthesized = '(' in context and ')' in context
        time_prefix = bool(re.search(
            r'(?:giai đoạn|thời kỳ|trị vì|sinh|mất|từ năm|đến năm)\s*[^.;:]{0,35}$',
            before,
        ))
        return parenthesized or time_prefix

    # Precision-first: số không có “năm”, niên đại trong ngoặc hoặc khoảng thời gian rõ ràng bị loại.
    return False


def year_context_supported(year: int, text: str) -> bool:
    body = normalize_keep_diacritics(text)
    for match in re.finditer(rf'(?<!\d){re.escape(str(int(year)))}(?!\d)', body):
        if _year_match_supported(int(year), body, match):
            return True
    return False


def extract_years(text: str) -> List[int]:
    body = normalize_keep_diacritics(text)
    found = set()
    for match in re.finditer(r'(?<!\d)(\d{1,4})(?!\d)', body):
        year = int(match.group(1))
        if _year_match_supported(year, body, match):
            found.add(year)
    return sorted(found)


# -------------------------
# Field-specific validation
# -------------------------
def exact_surface_supported(item: str, body: str) -> bool:
    item_norm = norm_surface(item)
    body_norm = norm_surface(body)
    if not item_norm:
        return False
    return bool(re.search(
        rf'(?<![a-z0-9]){re.escape(item_norm)}(?![a-z0-9])',
        body_norm,
    ))


def word_count(item: str) -> int:
    return len(norm_surface(item).split())


def occurrence_context(item: str, body: str, radius: int = 110) -> str:
    item_norm = norm_surface(item)
    body_norm = norm_surface(body)
    match = re.search(rf'(?<![a-z0-9]){re.escape(item_norm)}(?![a-z0-9])', body_norm)
    if not match:
        return ''
    return body_norm[max(0, match.start() - radius):min(len(body_norm), match.end() + radius)]


def titlecase_token_count(item: str) -> int:
    tokens = re.findall(r"[A-Za-zÀ-ỹĐđ][A-Za-zÀ-ỹĐđ'’-]*", clean_text(item))
    return sum(1 for token in tokens if token[:1].isupper() or token.isupper())


def _item_has_capitalized_neighbor(item: str, body: str) -> bool:
    """Loại mảnh tên bị cắt, ví dụ `Đức Vương` trong `Vũ Đức Vương`."""
    item_clean = clean_text(item)
    if not item_clean:
        return False

    pattern = re.compile(
        rf'(?<!\w){re.escape(item_clean)}(?!\w)',
        flags=re.UNICODE,
    )
    for match in pattern.finditer(clean_text(body)):
        before = body[max(0, match.start() - 40):match.start()]
        after = body[match.end():min(len(body), match.end() + 40)]

        prev_match = re.search(r"([^\W\d_][\w'’-]*)\s+$", before, flags=re.UNICODE)
        next_match = re.match(r"^\s+([^\W\d_][\w'’-]*)", after, flags=re.UNICODE)
        prev_token = prev_match.group(1) if prev_match else ''
        next_token = next_match.group(1) if next_match else ''

        prev_is_cap = bool(prev_token and prev_token[:1].isupper())
        next_is_cap = bool(next_token and next_token[:1].isupper())

        # Nếu candidate nằm sát một token viết hoa khác thì có thể là fragment của tên dài hơn.
        if prev_is_cap or next_is_cap:
            return True
    return False


def _role_only_person_phrase(item_norm: str) -> bool:
    tokens = item_norm.split()
    if not tokens:
        return True
    role_words = set()
    for value in PERSON_ROLE_ONLY_WORDS:
        role_words.update(value.split())
    return all(token in role_words for token in tokens)


def _location_occurrence_contexts(item: str, body: str, radius: int = 65) -> List[Tuple[str, str]]:
    item_norm = norm_surface(item)
    body_norm = norm_surface(body)
    contexts = []
    for match in re.finditer(rf'(?<![a-z0-9]){re.escape(item_norm)}(?![a-z0-9])', body_norm):
        before = body_norm[max(0, match.start() - radius):match.start()]
        after = body_norm[match.end():min(len(body_norm), match.end() + radius)]
        contexts.append((before, after))
    return contexts


def _location_occurrence_context(item: str, body: str, radius: int = 65) -> Tuple[str, str]:
    contexts = _location_occurrence_contexts(item, body, radius=radius)
    return contexts[0] if contexts else ('', '')


def _location_occurrence_contexts_diacritics(
    item: str, body: str, radius: int = 80
) -> List[Tuple[str, str]]:
    item_surface = normalize_keep_diacritics(item)
    body_surface = normalize_keep_diacritics(body)
    if not item_surface:
        return []
    contexts = []
    pattern = re.compile(
        rf'(?<![\wÀ-ỹĐđ]){re.escape(item_surface)}(?![\wÀ-ỹĐđ])',
        flags=re.UNICODE,
    )
    for match in pattern.finditer(body_surface):
        before = body_surface[max(0, match.start() - radius):match.start()]
        after = body_surface[match.end():min(len(body_surface), match.end() + radius)]
        contexts.append((before, after))
    return contexts


def _single_token_location_supported(item: str, body: str) -> bool:
    item_clean = clean_text(item)
    item_diac = normalize_keep_diacritics(item_clean)
    item_norm = norm_surface(item_clean)
    if item_diac in LOCATION_SINGLE_TOKEN_ALLOWLIST:
        return True

    # Nếu item nằm ngay trong title thì cho phép.
    title = clean_text(body).split('\n', 1)[0]
    if contains_surface_phrase(title, item_clean):
        return True

    before, after = _location_occurrence_context(item_clean, body, radius=45)
    if any(before.endswith(cue) for cue in LOCATION_GEO_CUES):
        return True
    if re.match(r'^\s*(?:tinh|thanh pho|huyen|quan|xa|phuong|song|nui|dao|vinh)\b', after):
        return True
    return False


def looks_like_person_name(item: str, body: str) -> bool:
    item_clean = clean_text(item)
    item_norm = norm_surface(item_clean)
    tokens = item_clean.split()
    count = len(tokens)

    if not exact_surface_supported(item_clean, body):
        return False
    if not 2 <= count <= 8 or any(ch.isdigit() for ch in item_clean):
        return False
    if item_norm in PERSON_FORBIDDEN_EXACT:
        return False
    if any(item_norm.startswith(prefix) for prefix in PERSON_FORBIDDEN_PREFIXES):
        return False
    if any(term in item_norm for term in PERSON_FORBIDDEN_SUBSTRINGS):
        return False
    if any(cue in item_norm for cue in PERIOD_CUES + EVENT_PREFIXES + DOCUMENT_CUES):
        return False
    if _role_only_person_phrase(item_norm):
        return False
    if _item_has_capitalized_neighbor(item_clean, body):
        return False

    first = normalize_keep_diacritics(tokens[0]).strip("'’-")
    caps = titlecase_token_count(item_clean)
    local = occurrence_context(item_clean, body)

    # Không nhận địa danh có họ người, ví dụ “động Đào Hoa”.
    body_surface = surface_text(body)
    item_surface = surface_text(item_clean)
    match_surface = re.search(rf'(?<!\w){re.escape(item_surface)}(?!\w)', body_surface)
    if match_surface:
        local_before = body_surface[max(0, match_surface.start() - 40):match_surface.start()]
        if re.search(
            r'\b(?:động|châu|tỉnh|huyện|xã|sông|núi|ải|phủ|thành|làng|quận|thị trấn|dân tộc|người)\s*$',
            local_before,
        ):
            return False

    # Tên Việt có họ quen thuộc.
    if first in VN_SURNAMES and caps >= 2:
        return True

    # Tên từ 3 token trở lên: vẫn phải có ít nhất 2 token viết hoa và không phải role phrase.
    if count >= 3 and caps >= 2:
        return True

    # Tên 2 token không có họ Việt chỉ nhận khi có ngữ cảnh tiểu sử rõ ràng.
    if count == 2 and caps == 2 and any(cue in local for cue in BIOGRAPHY_CUES):
        return True

    return False


def looks_like_event_name(item: str, body: str) -> bool:
    item_norm = norm_surface(item)
    if not exact_surface_supported(item, body) or not 2 <= word_count(item) <= 14:
        return False
    if any(mark in item for mark in ('.', ';', ':')):
        return False
    return any(item_norm.startswith(prefix) for prefix in EVENT_PREFIXES)


def looks_like_location(item: str, body: str) -> bool:
    item_clean = clean_text(item)
    item_norm = norm_surface(item_clean)
    count = word_count(item_clean)

    if not exact_surface_supported(item_clean, body) or not 1 <= count <= 8:
        return False
    if len(item_norm.replace(' ', '')) < 3:
        return False
    if item_norm in LOCATION_FORBIDDEN_EXACT:
        return False
    if any(item_norm.startswith(prefix) for prefix in LOCATION_FORBIDDEN_PREFIXES):
        return False
    if any(term in item_norm for term in LOCATION_FORBIDDEN_SUBSTRINGS):
        return False
    if any(item_norm.endswith(suffix) for suffix in LOCATION_FORBIDDEN_SUFFIXES):
        return False
    if item_norm in {norm_surface(label) for label in DYNASTY_SURFACE_PHRASES}:
        return False

    tokens_diac = set(surface_text(item_clean).split())
    # Các cụm mô tả như “Paris nhỏ ở Viễn Đông”, “cực bắc của Hà Giang”.
    if tokens_diac & LOCATION_INTERNAL_CONNECTORS:
        return False

    occurrence_contexts_diac = _location_occurrence_contexts_diacritics(
        item_clean, body, radius=80
    )
    if occurrence_contexts_diac:
        ethnic_flags = []
        for before, _after in occurrence_contexts_diac:
            # Danh sách tên dân tộc/tên tự gọi, ví dụ “Tên gọi khác: Mùn Di, Di, Màn Di...”.
            before_tail_diac = before[-90:]
            ethnonym_list = bool(re.search(
                r'(?:tên tự gọi|tên gọi khác|nhóm địa phương)\s*:\s*[^.;!?]{0,65}$',
                before_tail_diac,
                flags=re.IGNORECASE,
            ))

            # Chỉ xét cùng mệnh đề/câu phía trước candidate; dấu câu reset ngữ cảnh.
            same_clause_before = re.split(r'[.;:!?]', before)[-1][-70:]
            same_clause_norm = norm_surface(same_clause_before)
            # Chỉ coi là tên dân tộc khi cue nằm sát candidate hoặc dẫn một danh sách ngắn.
            # Không loại địa danh trong câu như “nhóm người Hoa ... tới Mỹ Tho”.
            direct_ethnic_cue = bool(re.search(
                r'(?:nguoi|dan toc|toc nguoi|nhom nguoi|sac toc)'
                r'(?:\s+[a-z]+){0,3}[,\s]*$',
                same_clause_norm,
            ))
            ethnic_flags.append(ethnonym_list or direct_ethnic_cue)
        # Chỉ loại khi mọi lần xuất hiện đều được dẫn bởi ngữ cảnh dân tộc/nhóm người.
        if all(ethnic_flags):
            return False

    # Một token phải có title/geographic cue hoặc nằm trong allowlist.
    if count == 1 and not _single_token_location_supported(item_clean, body):
        return False

    # Tên địa danh thường có ít nhất một token viết hoa hoặc là viết tắt.
    return titlecase_token_count(item_clean) >= 1 or item_clean.isupper()


def looks_like_document_name(item: str, body: str) -> bool:
    item_norm = norm_surface(item)
    if not exact_surface_supported(item, body) or not 2 <= word_count(item) <= 16:
        return False
    if item_norm in GENERIC_DOCUMENTS or item_norm.endswith(' sac lenh so'):
        return False
    if item_norm == 'sac lenh so' or re.fullmatch(r'sac lenh so', item_norm):
        return False
    if item_norm.startswith('sac lenh so'):
        # Phải có số hiệu sau chữ “số”.
        return bool(re.search(r'\bsac lenh so\s+[0-9a-z]', item_norm))
    return any(cue in item_norm for cue in DOCUMENT_CUES)


def looks_like_period(item: str, body: str) -> bool:
    item_norm = norm_surface(item)
    if not exact_surface_supported(item, body) or not 2 <= word_count(item) <= 10:
        return False
    if any(role in item_norm for role in PERSON_ROLE_CUES):
        return False
    if any(item_norm.startswith(prefix) for prefix in PERSON_FORBIDDEN_PREFIXES):
        return False
    return any(cue in item_norm for cue in PERIOD_CUES)


def field_supported(field: str, item: str, body: str) -> bool:
    item = clean_text(item).strip(' ,;:.–—-')
    if field == 'people':
        return looks_like_person_name(item, body)
    if field == 'events':
        return looks_like_event_name(item, body)
    if field == 'locations':
        return looks_like_location(item, body)
    if field == 'documents':
        return looks_like_document_name(item, body)
    if field == 'periods':
        return looks_like_period(item, body)
    return False


def rule_metadata(chunk: Dict[str, Any]) -> Dict[str, Any]:
    title = clean_text(chunk.get('title', ''))
    text = clean_text(chunk.get('text', ''))
    body = f'{title}\n{text}'
    return {
        'years': extract_years(body),
        'dynasties': extract_dynasties(body, max_items=8),
        'topics': extract_rule_labels(body, TOPIC_PATTERNS, max_items=4),
        'content_facets': extract_rule_labels(body, FACET_PATTERNS, max_items=3),
    }


def source_sha1(chunk: Dict[str, Any]) -> str:
    payload = (
        str(chunk.get('chunk_id', '')) + '\n'
        + clean_text(chunk.get('title', '')) + '\n'
        + clean_text(chunk.get('text', ''))
    )
    return hashlib.sha1(payload.encode('utf-8')).hexdigest()


# =========================
# Regression tests cho các lỗi đã gặp
# =========================
assert extract_years('Lưu lượng 20–500 m³/s; tầng nước 60–90 m và 170–200 m.') == []
assert extract_years('Phát sóng từ những năm 60; doanh thu chiếm 60–70%.') == []
assert extract_years('Cách nay 50–60 triệu năm; di cư cách đây 200–300 năm.') == []
assert extract_years('Năm 40, cuộc khởi nghĩa Hai Bà Trưng bùng nổ; năm 938 có trận Bạch Đằng.') == [40, 938]
assert extract_years('Năm 1698 lập phủ Gia Định. Paul Coffyn (1810–1871).') == [1698, 1810, 1871]

assert extract_dynasties('Thành phố có 5 nhà hộ sinh.') == []
assert extract_dynasties('Nhà Hồ tồn tại trong lịch sử Việt Nam.') == ['Nhà Hồ']

_test_body = (
    'Nguyễn Trãi thay Lê Lợi soạn Bình Ngô đại cáo. '
    'Người Việt sinh sống tại đây. Thủ tướng Chính phủ chủ trì cuộc họp. '
    'Hội đồng Chính phủ ban hành Sắc lệnh số 78-SL. '
    'Khởi nghĩa Lam Sơn diễn ra ở Lam Sơn. Đổi trấn làm châu là một hành động hành chính. '
    'Sài Gòn là địa danh; Phòng Nam Bộ Trung ương là cơ quan.'
)
assert field_supported('people', 'Nguyễn Trãi', _test_body)
assert not field_supported('people', 'Người Việt', _test_body)
assert not field_supported('people', 'Thủ tướng Chính phủ', _test_body)
assert not field_supported('people', 'Hội đồng Chính phủ', _test_body)
assert field_supported('events', 'Khởi nghĩa Lam Sơn', _test_body)
assert not field_supported('events', 'đổi trấn làm châu', _test_body)
assert field_supported('documents', 'Bình Ngô đại cáo', _test_body)
assert field_supported('documents', 'Sắc lệnh số 78-SL', _test_body)
assert not field_supported('documents', 'Sắc lệnh số', _test_body)
assert field_supported('locations', 'Sài Gòn', _test_body)
assert not field_supported('locations', 'Phòng Nam Bộ Trung ương', _test_body)

# Regression tests v4 cho lỗi people/locations đã gặp trong sample 50.
_people_patch_body = (
    'Quan Thác đông tiết độ cai quản vùng này. Vũ Đức Vương được nhắc đến. '
    'Chủ tịch Chính phủ ký văn bản. Nguyễn Trãi và Paul Coffyn đều là nhân vật lịch sử.'
)
assert not field_supported('people', 'Quan Thác đông', _people_patch_body)
assert not field_supported('people', 'Đức Vương', _people_patch_body)
assert not field_supported('people', 'Chủ tịch Chính phủ', _people_patch_body)
assert field_supported('people', 'Nguyễn Trãi', _people_patch_body)

_location_patch_body = (
    'Sài Gòn từng được gọi là Paris nhỏ ở Viễn Đông. Sài Gòn mở rộng về phía nam. '
    'Viện Khoa học Thủy lợi miền Nam là một cơ quan. Kinh sư là cách gọi chung. '
    'Khu vực cực bắc của Hà Giang có các nhóm người Mùn Di, Di và Màn Di. '
    'Thành phố Hồ Chí Minh, Hà Giang và Lào Cai là địa danh.'
)
assert not field_supported('locations', 'Paris nhỏ ở Viễn Đông', _location_patch_body)
assert not field_supported('locations', 'Sài Gòn mở rộng', _location_patch_body)
assert not field_supported('locations', 'Viện Khoa học Thủy lợi miền Nam', _location_patch_body)
assert not field_supported('locations', 'Kinh sư', _location_patch_body)
assert not field_supported('locations', 'cực bắc của Hà Giang', _location_patch_body)
assert not field_supported('locations', 'Di', _location_patch_body)
assert not field_supported('locations', 'Mùn Di', _location_patch_body)
assert field_supported('locations', 'Hà Giang', _location_patch_body)
assert field_supported('locations', 'Lào Cai', _location_patch_body)
assert field_supported('locations', 'Việt Nam', _location_patch_body + ' Việt Nam là một quốc gia.')
assert field_supported('locations', 'An Nam', _location_patch_body + ' An Nam là một tên gọi lịch sử.')

_ethnonym_list_body = (
    'Tên tự gọi: Lô Lô. Tên gọi khác: Mùn Di, Di, Màn Di, La La, Qua La. '
    'Lào Cai và Hà Giang là địa danh.'
)
for _bad_ethnonym in ['Mùn Di', 'Di', 'Màn Di', 'La La', 'Qua La']:
    assert not field_supported('locations', _bad_ethnonym, _ethnonym_list_body)
assert field_supported('locations', 'Lào Cai', _ethnonym_list_body)
assert field_supported('locations', 'Hà Giang', _ethnonym_list_body)

print('Helpers v4 loaded; regression tests passed.')
for c in chunks[:3]:
    print(c['chunk_id'], rule_metadata(c))


Helpers v4 loaded; regression tests passed.
hf_wikipedia_thành_phố_hồ_chí_minh_0000_2da892be6bd2 {'years': [1297, 1698, 2011, 2018, 2019, 2020, 2021], 'dynasties': ['Chúa Nguyễn'], 'topics': ['kinh tế', 'văn hóa', 'xã hội', 'giáo dục'], 'content_facets': ['kết quả']}
hf_wikipedia_thành_phố_hồ_chí_minh_0001_ed1d52d504fc {'years': [1674, 1698, 1700, 1731, 1747, 1772, 1776, 1887, 1901, 1931, 1941, 1946, 1954, 1975, 1976], 'dynasties': ['Chúa Nguyễn', 'Việt Nam Dân chủ Cộng hòa', 'Việt Nam Cộng hòa', 'Cộng hòa Miền Nam Việt Nam'], 'topics': ['ngoại giao', 'thống nhất', 'thuộc địa', 'văn kiện'], 'content_facets': []}
hf_wikipedia_thành_phố_hồ_chí_minh_0002_4637b9fa546a {'years': [1620, 1623, 1679, 1976], 'dynasties': ['Chúa Nguyễn', 'Nhà Nguyễn', 'Đế quốc Khmer', 'Việt Nam Dân chủ Cộng hòa', 'Cộng hòa Miền Nam Việt Nam'], 'topics': ['kinh tế', 'văn hóa', 'xã hội', 'thống nhất'], 'content_facets': []}


In [5]:

# Cell 5 — Load Qwen extractor (bỏ qua nếu rules_only)

model = None
tokenizer = None

if EXTRACTION_MODE == 'hybrid':
    from transformers import (
        AutoTokenizer,
        AutoModelForCausalLM,
        BitsAndBytesConfig,
    )

    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
    )
    tokenizer.padding_side = 'left'
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    model_kwargs = {
        'device_map': 'auto',
        'trust_remote_code': True,
        'torch_dtype': torch.bfloat16 if torch.cuda.is_available() else torch.float32,
    }

    if LOAD_IN_4BIT and torch.cuda.is_available():
        model_kwargs['quantization_config'] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_quant_type='nf4',
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_use_double_quant=True,
        )

    model = AutoModelForCausalLM.from_pretrained(MODEL_ID, **model_kwargs)
    model.eval()
    print('Đã load extractor:', MODEL_ID)
else:
    print('EXTRACTION_MODE=rules_only: không load LLM.')


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Đã load extractor: Qwen/Qwen2.5-3B-Instruct


In [6]:
# Cell 6 — Prompt trích entity bề mặt và parser JSON

SYSTEM_PROMPT = """Bạn là bộ trích xuất metadata precision-first cho kho tư liệu lịch sử Việt Nam.
Chỉ dùng nội dung xuất hiện trực tiếp trong TIÊU ĐỀ và NỘI DUNG được cung cấp.
Không dùng kiến thức bên ngoài, không suy đoán, không viết câu giải thích.
Nếu không chắc chắn, phải để mảng rỗng. Chỉ trả về đúng một JSON object hợp lệ.
"""

JSON_SCHEMA_EXAMPLE = {
    'years': [],
    'people': [],
    'events': [],
    'locations': [],
    'documents': [],
    'periods': [],
}


def build_extraction_prompt(chunk: Dict[str, Any]) -> str:
    title = clean_text(chunk.get('title', ''))
    text = clean_text(chunk.get('text', ''))[:MAX_INPUT_CHARS]

    return f"""Trích metadata bề mặt cho chunk dưới đây.

QUY TẮC BẮT BUỘC:
- years: chỉ niên đại cụ thể gắn với sự kiện, thành lập, trị vì, sinh-mất, ký kết. Không lấy thập niên như “những năm 60”; không lấy số đo, dân số, diện tích, phần trăm, tần số, khoảng cách, độ sâu, tuổi hoặc khoảng thời gian như “200–300 năm trước”.
- people: chỉ TÊN CÁ NHÂN ĐẦY ĐỦ. Không lấy dân tộc/nhóm người (“Người Việt”), chức danh đứng riêng (“Thủ tướng Chính phủ”, “Quan Thác đông”), tổ chức, địa danh hoặc mảnh tên bị cắt (“Đức Vương” từ “Vũ Đức Vương”). Trả tên riêng, không kèm chức danh.
- events: chỉ TÊN SỰ KIỆN bắt đầu bằng dạng như Khởi nghĩa, Chiến dịch, Trận, Chiến tranh, Cách mạng, Phong trào, Tổng tiến công, Hội nghị, Biến cố, Nạn đói. Không lấy câu hành động như “đổi trấn làm châu”.
- locations: chỉ tên riêng của địa danh/quốc gia/sông/vùng. Không lấy cơ quan (“Viện Khoa học…”), doanh nghiệp, chức danh, triều đại, biệt danh (“Paris nhỏ ở Viễn Đông”), mô tả mở rộng (“Sài Gòn mở rộng”), mô tả hướng (“cực bắc của Hà Giang”) hoặc tên dân tộc (“Mùn Di”, “Di”).
- documents: chỉ tên văn kiện/tác phẩm đầy đủ và phân biệt được. “Sắc lệnh số” đứng riêng là không hợp lệ; phải có số hiệu hoặc tên đầy đủ.
- periods: chỉ tên giai đoạn/thời kỳ được nêu trực tiếp, ví dụ “thời Pháp thuộc”; không đưa tên người, chức danh hoặc tổ chức.
- Mỗi item phải xuất hiện nguyên cụm trong TIÊU ĐỀ hoặc NỘI DUNG.
- Tối đa: years 15; people 8; events 6; locations 8; documents 6; periods 4.
- Nếu không có bằng chứng chắc chắn, trả về [].

JSON cần trả:
{json.dumps(JSON_SCHEMA_EXAMPLE, ensure_ascii=False)}

CHUNK_ID: {chunk.get('chunk_id', '')}
TIÊU ĐỀ: {title}
NỘI DUNG:
{text}
"""


def parse_json_object(raw: str) -> Dict[str, Any]:
    raw = clean_text(raw)
    raw = re.sub(r'^```(?:json)?\s*', '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'\s*```$', '', raw)

    start = raw.find('{')
    end = raw.rfind('}')
    if start >= 0 and end > start:
        raw = raw[start:end + 1]

    repaired = repair_json(raw, return_objects=False)
    obj = json.loads(repaired)
    if not isinstance(obj, dict):
        raise ValueError('LLM output không phải JSON object.')
    return obj


In [7]:

# Cell 7 — Batch generation metadata bằng Qwen

@torch.inference_mode()
def llm_extract_batch(batch_chunks: List[Dict[str, Any]]) -> List[Tuple[Dict[str, Any], str]]:
    if EXTRACTION_MODE != 'hybrid':
        return [({}, '') for _ in batch_chunks]

    rendered = []
    for chunk in batch_chunks:
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user', 'content': build_extraction_prompt(chunk)},
        ]
        rendered.append(
            tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        )

    inputs = tokenizer(
        rendered,
        return_tensors='pt',
        padding=True,
        truncation=True,
        max_length=8192,
    )

    device = model.get_input_embeddings().weight.device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    input_len = inputs['input_ids'].shape[1]

    outputs = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
        use_cache=True,
    )

    generated = outputs[:, input_len:]
    raw_texts = tokenizer.batch_decode(generated, skip_special_tokens=True)

    parsed = []
    for raw in raw_texts:
        parsed.append((parse_json_object(raw), raw))
    return parsed


In [8]:
# Cell 8 — Chuẩn hóa, validator, merge và final sanitizer


def as_string_list(value: Any, max_items: int) -> List[str]:
    if value is None:
        return []
    if isinstance(value, str):
        value = [value]
    if not isinstance(value, list):
        return []
    return [
        item for item in stable_unique(value)
        if 1 <= len(item) <= 120
    ][:max_items]


def as_year_list(value: Any, max_items: int = 15) -> List[int]:
    if value is None:
        return []
    if isinstance(value, (str, int, float)):
        value = [value]
    if not isinstance(value, list):
        return []

    years = []
    for item in value:
        match = re.fullmatch(r'\s*(\d{1,4})\s*', str(item))
        if not match:
            continue
        year = int(match.group(1))
        if valid_year_value(year):
            years.append(year)
    return sorted(set(years))[:max_items]


def normalize_llm_metadata(raw_obj: Dict[str, Any]) -> Dict[str, Any]:
    return {
        'years': as_year_list(raw_obj.get('years', []), 15),
        'people': as_string_list(raw_obj.get('people', []), 8),
        'events': as_string_list(raw_obj.get('events', []), 6),
        'locations': as_string_list(raw_obj.get('locations', []), 8),
        'documents': as_string_list(raw_obj.get('documents', []), 6),
        'periods': as_string_list(raw_obj.get('periods', []), 4),
    }


def derive_topics(rule_topics: List[str], accepted: Dict[str, List[str]]) -> List[str]:
    topics = list(rule_topics)
    if accepted.get('people'):
        topics.append('nhân vật lịch sử')
    if accepted.get('documents'):
        topics.append('văn kiện')

    event_text = norm_surface(' '.join(accepted.get('events', [])))
    if 'khoi nghia' in event_text:
        topics.append('khởi nghĩa')
    if 'khang chien' in event_text:
        topics.append('kháng chiến')
    if 'cach mang' in event_text:
        topics.append('cách mạng')
    return stable_unique(topics)[:4]


def _quality_flags(record: Dict[str, Any], rejected: Dict[str, List[Any]], status: str) -> List[str]:
    flags = []
    if status != 'ok':
        flags.append(status)
    if not any([
        record.get('years'), record.get('people'), record.get('events'),
        record.get('locations'), record.get('documents'), record.get('periods'),
        record.get('dynasties'), record.get('topics'), record.get('content_facets'),
    ]):
        flags.append('metadata_empty')
    if sum(len(v) for v in rejected.values()) >= 4:
        flags.append('many_rejected_items')
    return stable_unique(flags)


def build_metadata_record(
    chunk: Dict[str, Any],
    llm_obj: Optional[Dict[str, Any]] = None,
    raw_llm_output: str = '',
    extraction_status: str = 'ok',
    llm_error: str = '',
) -> Dict[str, Any]:
    title = clean_text(chunk.get('title', ''))
    text = clean_text(chunk.get('text', ''))
    body = f'{title}\n{text}'

    rules = rule_metadata(chunk)
    llm_meta = normalize_llm_metadata(llm_obj or {})

    accepted: Dict[str, List[str]] = {}
    rejected: Dict[str, List[Any]] = {}

    for field in SURFACE_FIELDS:
        keep, drop = [], []
        for item in llm_meta.get(field, []):
            if field_supported(field, item, body):
                keep.append(item)
            else:
                drop.append(item)
        accepted[field] = stable_unique(keep)
        if drop:
            rejected[field] = stable_unique(drop)

    llm_years_keep, llm_years_drop = [], []
    for year in llm_meta.get('years', []):
        if year_context_supported(year, body):
            llm_years_keep.append(year)
        else:
            llm_years_drop.append(year)
    if llm_years_drop:
        rejected['years'] = sorted(set(llm_years_drop))

    years = sorted(set(rules.get('years', []) + llm_years_keep))[:20]
    topics = derive_topics(rules.get('topics', []), accepted)
    facets = stable_unique(rules.get('content_facets', []))[:3]

    record = {
        'chunk_id': str(chunk.get('chunk_id', '')).strip(),
        'metadata_version': METADATA_VERSION,
        'validator_signature': VALIDATOR_SIGNATURE,
        'extraction_method': EXTRACTION_MODE,
        'extraction_status': extraction_status,
        'source_sha1': source_sha1(chunk),
        'years': years,
        'people': accepted.get('people', []),
        'events': accepted.get('events', []),
        'locations': accepted.get('locations', []),
        'documents': accepted.get('documents', []),
        'periods': accepted.get('periods', []),
        'dynasties': rules.get('dynasties', []),
        'topics': topics,
        'content_facets': facets,
        'rejected_items': rejected,
        'quality_flags': [],
        'extracted_at_utc': datetime.now(timezone.utc).isoformat(),
    }
    record['quality_flags'] = _quality_flags(record, rejected, extraction_status)

    if llm_error:
        record['llm_error'] = clean_text(llm_error)[:500]
    if KEEP_RAW_LLM_OUTPUT and raw_llm_output:
        record['raw_llm_output'] = raw_llm_output
    return record


def final_sanitize_record(record: Dict[str, Any], chunk: Dict[str, Any]) -> Dict[str, Any]:
    """Sanitize lại toàn bộ record trước khi ghi file chính thức."""
    body = f"{clean_text(chunk.get('title', ''))}\n{clean_text(chunk.get('text', ''))}"
    rejected = dict(record.get('rejected_items', {}) or {})

    cleaned = dict(record)
    cleaned['chunk_id'] = str(chunk.get('chunk_id', '')).strip()
    cleaned['metadata_version'] = METADATA_VERSION
    cleaned['validator_signature'] = VALIDATOR_SIGNATURE
    cleaned['source_sha1'] = source_sha1(chunk)

    years_keep = [int(y) for y in record.get('years', []) if year_context_supported(int(y), body)]
    years_drop = [y for y in record.get('years', []) if int(y) not in set(years_keep)]
    cleaned['years'] = sorted(set(years_keep))[:20]
    if years_drop:
        rejected['years_final'] = sorted(set(years_drop))

    for field, max_items in [('people', 8), ('events', 6), ('locations', 8), ('documents', 6), ('periods', 4)]:
        keep, drop = [], []
        for item in record.get(field, []) or []:
            if field_supported(field, item, body):
                keep.append(item)
            else:
                drop.append(item)
        cleaned[field] = stable_unique(keep)[:max_items]
        if drop:
            rejected[f'{field}_final'] = stable_unique(drop)

    # Recompute deterministic fields từ raw text; không tin checkpoint cũ.
    rules = rule_metadata(chunk)
    cleaned['dynasties'] = rules['dynasties']
    cleaned['content_facets'] = rules['content_facets'][:3]
    cleaned['topics'] = derive_topics(rules['topics'], cleaned)
    cleaned['rejected_items'] = rejected
    cleaned['quality_flags'] = _quality_flags(
        cleaned,
        rejected,
        cleaned.get('extraction_status', 'ok'),
    )
    return cleaned


# Unit test end-to-end.
_demo_chunk = {
    'chunk_id': 'demo_001',
    'title': 'Bình Ngô đại cáo',
    'text': (
        'Sau thắng lợi của Khởi nghĩa Lam Sơn, Nguyễn Trãi thay Lê Lợi '
        'soạn Bình Ngô đại cáo năm 1428. Dân số là 9.166.800 người. '
        'Thành phố có 5 nhà hộ sinh. Hội đồng Chính phủ ban hành Sắc lệnh số 78-SL.'
    ),
}
_demo_llm = {
    'years': [1428, 166, 800, 60],
    'people': ['Nguyễn Trãi', 'Lê Lợi', 'Người Việt', 'Hội đồng Chính phủ'],
    'events': ['Khởi nghĩa Lam Sơn', 'đổi trấn làm châu'],
    'locations': ['Sài Gòn', 'Phòng Nam Bộ Trung ương'],
    'documents': ['Bình Ngô đại cáo', 'Sắc lệnh số', 'Sắc lệnh số 78-SL'],
    'periods': ['Thủ tướng Nguyễn Cao Kỳ'],
}
_demo_meta = final_sanitize_record(build_metadata_record(_demo_chunk, _demo_llm), _demo_chunk)
assert _demo_meta['years'] == [1428], _demo_meta['years']
assert _demo_meta['people'] == ['Nguyễn Trãi', 'Lê Lợi'], _demo_meta['people']
assert _demo_meta['events'] == ['Khởi nghĩa Lam Sơn'], _demo_meta['events']
assert _demo_meta['documents'] == ['Bình Ngô đại cáo', 'Sắc lệnh số 78-SL'], _demo_meta['documents']
assert _demo_meta['periods'] == [], _demo_meta['periods']
assert _demo_meta['dynasties'] == [], _demo_meta['dynasties']
print(json.dumps(_demo_meta, ensure_ascii=False, indent=2))


{
  "chunk_id": "demo_001",
  "metadata_version": "v4.0-final-people-location-strict",
  "validator_signature": "phase8_v4_people_location_20260731_a",
  "extraction_method": "hybrid",
  "extraction_status": "ok",
  "source_sha1": "706a2bf30c9eb535f8d79f3002903b5cd1307e73",
  "years": [
    1428
  ],
  "people": [
    "Nguyễn Trãi",
    "Lê Lợi"
  ],
  "events": [
    "Khởi nghĩa Lam Sơn"
  ],
  "locations": [],
  "documents": [
    "Bình Ngô đại cáo",
    "Sắc lệnh số 78-SL"
  ],
  "periods": [],
  "dynasties": [],
  "topics": [
    "xã hội",
    "khởi nghĩa",
    "văn kiện",
    "nhân vật lịch sử"
  ],
  "content_facets": [],
  "rejected_items": {
    "people": [
      "Người Việt",
      "Hội đồng Chính phủ"
    ],
    "events": [
      "đổi trấn làm châu"
    ],
    "locations": [
      "Sài Gòn",
      "Phòng Nam Bộ Trung ương"
    ],
    "documents": [
      "Sắc lệnh số"
    ],
    "periods": [
      "Thủ tướng Nguyễn Cao Kỳ"
    ],
    "years": [
      60,
      166,
      800


In [9]:
# ============================================================
# REPAIR CHECKPOINT v4
# Bỏ dòng JSON hỏng + deduplicate theo chunk_id
# ============================================================

import json
import os
import shutil
from pathlib import Path

checkpoint_path = Path(CHECKPOINT_PATH)

backup_path = checkpoint_path.with_suffix(
    checkpoint_path.suffix + ".corrupt_backup"
)

repaired_path = checkpoint_path.with_suffix(
    checkpoint_path.suffix + ".repaired"
)

valid_by_id = {}
bad_lines = []
valid_lines = 0

print("Đang kiểm tra checkpoint:", checkpoint_path)

with checkpoint_path.open(
    "r",
    encoding="utf-8",
    errors="replace"
) as f:

    for line_no, line in enumerate(f, start=1):

        raw = line.strip()

        if not raw:
            continue

        try:
            obj = json.loads(raw)

            if not isinstance(obj, dict):
                raise ValueError("Record không phải JSON object")

            chunk_id = str(obj.get("chunk_id", "")).strip()

            if not chunk_id:
                raise ValueError("Thiếu chunk_id")

            # Giữ record cuối nếu một chunk_id xuất hiện nhiều lần.
            valid_by_id[chunk_id] = obj
            valid_lines += 1

        except Exception as exc:

            bad_lines.append({
                "line": line_no,
                "error": str(exc),
                "preview": raw[:300],
            })


print("\n===== SCAN RESULT =====")
print("Valid JSON lines :", valid_lines)
print("Unique chunk_ids :", len(valid_by_id))
print("Bad lines        :", len(bad_lines))

if bad_lines:
    print("\nBad line samples:")

    for x in bad_lines[:20]:
        print(
            f"Line {x['line']}: "
            f"{x['error']}\n"
            f"{x['preview'][:200]}\n"
        )


# ------------------------------------------------------------
# Compare với corpus
# ------------------------------------------------------------

corpus_ids = {
    str(chunk["chunk_id"])
    for chunk in chunks
}

checkpoint_ids = set(valid_by_id)

missing_ids = corpus_ids - checkpoint_ids
extra_ids = checkpoint_ids - corpus_ids

print("\n===== CORPUS COMPARISON =====")
print("Corpus chunks :", len(corpus_ids))
print("Checkpoint IDs:", len(checkpoint_ids))
print("Missing       :", len(missing_ids))
print("Extra         :", len(extra_ids))

if missing_ids:
    print("Ví dụ missing:", list(missing_ids)[:10])


# ------------------------------------------------------------
# Write repaired checkpoint atomically
# ------------------------------------------------------------

with repaired_path.open(
    "w",
    encoding="utf-8"
) as f:

    for obj in valid_by_id.values():
        f.write(
            json.dumps(
                obj,
                ensure_ascii=False
            ) + "\n"
        )

    f.flush()
    os.fsync(f.fileno())


# Backup file hỏng trước.
if not backup_path.exists():
    shutil.copy2(
        checkpoint_path,
        backup_path
    )

# Promote repaired checkpoint.
os.replace(
    repaired_path,
    checkpoint_path
)

print("\n✅ CHECKPOINT ĐÃ REPAIR")
print("Backup :", backup_path)
print("Current:", checkpoint_path)

Đang kiểm tra checkpoint: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata/vn_history_rag_chunk_metadata.v4.checkpoint.jsonl

===== SCAN RESULT =====
Valid JSON lines : 58601
Unique chunk_ids : 58601
Bad lines        : 0

===== CORPUS COMPARISON =====
Corpus chunks : 58603
Checkpoint IDs: 58601
Missing       : 2
Extra         : 0
Ví dụ missing: ['hf_wikipedia_ảnh_hưởng_xã_hội_của_đại_dịch_covid-19_tại_việt_nam_0006_27a58e7c80d8', 'hf_wikipedia_ảnh_hưởng_xã_hội_của_đại_dịch_covid-19_tại_việt_nam_0007_694faa4d5ff3']

✅ CHECKPOINT ĐÃ REPAIR
Backup : /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata/vn_history_rag_chunk_metadata.v4.checkpoint.jsonl.corrupt_backup
Current: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata/vn_history_rag_chunk_metadata.v4.checkpoint.jsonl


In [10]:
# Cell 9 — Checkpoint, resume và vòng lặp xử lý toàn bộ corpus


def append_jsonl(path: Path, records: List[Dict[str, Any]]) -> None:
    if not records:
        return
    with Path(path).open('a', encoding='utf-8') as file:
        for obj in records:
            file.write(json.dumps(obj, ensure_ascii=False) + '\n')
        file.flush()
        os.fsync(file.fileno())


def load_checkpoint(path: Path) -> Dict[str, Dict[str, Any]]:
    if not path.exists():
        return {}
    by_id = {}
    for obj in read_jsonl(path):
        chunk_id = str(obj.get('chunk_id', '')).strip()
        if chunk_id:
            by_id[chunk_id] = obj  # Record cuối cùng thắng nếu có trùng.
    return by_id


if FORCE_REBUILD:
    for path in (CHECKPOINT_PATH, ERROR_PATH):
        if path.exists():
            path.unlink()
            print('Đã xóa:', path)

completed = load_checkpoint(CHECKPOINT_PATH) if RESUME else {}

pending = []
for chunk in chunks:
    chunk_id = str(chunk['chunk_id'])
    old = completed.get(chunk_id)
    if (
        old
        and old.get('source_sha1') == source_sha1(chunk)
        and old.get('metadata_version') == METADATA_VERSION
        and old.get('validator_signature') == VALIDATOR_SIGNATURE
        and old.get('extraction_method') == EXTRACTION_MODE
    ):
        continue
    pending.append(chunk)

print(f'Checkpoint v4 đã có: {len(completed):,}')
print(f'Còn cần xử lý: {len(pending):,}/{len(chunks):,}')

error_buffer = []
processed_batches = 0

for start in tqdm(range(0, len(pending), BATCH_SIZE), desc='Extract metadata v4'):
    batch = pending[start:start + BATCH_SIZE]

    try:
        if EXTRACTION_MODE == 'hybrid':
            batch_outputs = llm_extract_batch(batch)
            if len(batch_outputs) != len(batch):
                raise RuntimeError(
                    f'LLM trả {len(batch_outputs)} output cho batch {len(batch)} chunk.'
                )
        else:
            batch_outputs = [({}, '') for _ in batch]

        new_records = []
        for chunk, (llm_obj, raw_text) in zip(batch, batch_outputs):
            record = build_metadata_record(chunk, llm_obj, raw_text)
            completed[record['chunk_id']] = record
            new_records.append(record)

        append_jsonl(CHECKPOINT_PATH, new_records)
        processed_batches += 1

        if processed_batches % SAVE_EVERY == 0:
            valid_completed = sum(
                rec.get('metadata_version') == METADATA_VERSION
                for rec in completed.values()
            )
            print(f'Checkpoint v4: {valid_completed:,}/{len(chunks):,}')

    except Exception as batch_exc:
        print(f'Batch lỗi tại start={start}: {type(batch_exc).__name__}: {batch_exc}')
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        # Thử lại từng chunk; nếu vẫn lỗi thì tạo rules-only fallback.
        for chunk in batch:
            try:
                if EXTRACTION_MODE == 'hybrid':
                    llm_obj, raw_text = llm_extract_batch([chunk])[0]
                else:
                    llm_obj, raw_text = {}, ''

                record = build_metadata_record(chunk, llm_obj, raw_text)

            except Exception as inner_exc:
                error = {
                    'chunk_id': str(chunk.get('chunk_id', '')),
                    'error_type': type(inner_exc).__name__,
                    'error': str(inner_exc),
                    'at_utc': datetime.now(timezone.utc).isoformat(),
                }
                error_buffer.append(error)
                append_jsonl(ERROR_PATH, [error])

                if not RULES_FALLBACK_ON_LLM_ERROR:
                    continue

                record = build_metadata_record(
                    chunk,
                    llm_obj={},
                    raw_llm_output='',
                    extraction_status='rules_fallback',
                    llm_error=f'{type(inner_exc).__name__}: {inner_exc}',
                )

            completed[record['chunk_id']] = record
            append_jsonl(CHECKPOINT_PATH, [record])

print('Hoàn tất vòng lặp.')
print('Metadata trong bộ nhớ:', len(completed))
print('Số lỗi LLM có fallback:', len(error_buffer))


Checkpoint v4 đã có: 58,601
Còn cần xử lý: 2/58,603


Extract metadata v4:   0%|          | 0/1 [00:00<?, ?it/s]

Hoàn tất vòng lặp.
Metadata trong bộ nhớ: 58603
Số lỗi LLM có fallback: 0


In [11]:
# Cell 10 — Final sanitize, finalize atomically và merge bằng chunk_id

# Bảo đảm mọi chunk đều có metadata. Chunk thiếu dùng rule-only fallback.
missing_ids = []
for chunk in chunks:
    chunk_id = str(chunk['chunk_id'])
    record = completed.get(chunk_id)
    if not (
        record
        and record.get('metadata_version') == METADATA_VERSION
        and record.get('validator_signature') == VALIDATOR_SIGNATURE
        and record.get('source_sha1') == source_sha1(chunk)
    ):
        missing_ids.append(chunk_id)
        fallback = build_metadata_record(
            chunk,
            llm_obj={},
            extraction_status='rules_fallback_finalize',
            llm_error='Không có record hybrid v4 hợp lệ khi finalize.',
        )
        completed[chunk_id] = fallback
        append_jsonl(CHECKPOINT_PATH, [fallback])

if missing_ids:
    print(f'Đã tạo rules-only fallback cho {len(missing_ids):,} chunk còn thiếu.')
    print('Ví dụ:', missing_ids[:10])

# Sanitize lại tất cả record, kể cả record đến từ checkpoint.
ordered_metadata = []
for chunk in tqdm(chunks, desc='Final sanitize v4'):
    chunk_id = str(chunk['chunk_id'])
    cleaned = final_sanitize_record(completed[chunk_id], chunk)
    completed[chunk_id] = cleaned
    ordered_metadata.append(cleaned)

if len(ordered_metadata) != len(chunks):
    raise RuntimeError('Số metadata không bằng số chunk.')

metadata_ids = [record['chunk_id'] for record in ordered_metadata]
corpus_ids = [str(chunk['chunk_id']) for chunk in chunks]
if metadata_ids != corpus_ids:
    raise RuntimeError('Thứ tự/ID metadata không khớp corpus.')
if len(set(metadata_ids)) != len(metadata_ids):
    raise RuntimeError('Metadata có chunk_id trùng.')

# Pre-write strict validation.
prewrite_errors = []
for chunk, record in zip(chunks, ordered_metadata):
    body = f"{clean_text(chunk.get('title', ''))}\n{clean_text(chunk.get('text', ''))}"
    for year in record.get('years', []):
        if not year_context_supported(year, body):
            prewrite_errors.append(f"{record['chunk_id']}: invalid year {year}")
    for field in SURFACE_FIELDS:
        for item in record.get(field, []):
            if not field_supported(field, item, body):
                prewrite_errors.append(f"{record['chunk_id']}: invalid {field}={item}")
    expected_dynasties = extract_dynasties(body, max_items=8)
    if record.get('dynasties', []) != expected_dynasties:
        prewrite_errors.append(f"{record['chunk_id']}: dynasty mismatch")
    if len(prewrite_errors) >= 100:
        break

if prewrite_errors:
    raise RuntimeError('Pre-write QC thất bại:\n' + '\n'.join(prewrite_errors[:100]))

# Ghi file tạm rồi replace để tránh output dở dang.
metadata_tmp = METADATA_PATH.with_suffix(METADATA_PATH.suffix + '.building')
enriched_tmp = ENRICHED_PATH.with_suffix(ENRICHED_PATH.suffix + '.building')

with metadata_tmp.open('w', encoding='utf-8') as file:
    for record in ordered_metadata:
        file.write(json.dumps(record, ensure_ascii=False) + '\n')
    file.flush()
    os.fsync(file.fileno())

metadata_by_id = {record['chunk_id']: record for record in ordered_metadata}

with enriched_tmp.open('w', encoding='utf-8') as file:
    for chunk in chunks:
        chunk_id = str(chunk['chunk_id'])
        enriched = dict(chunk)
        enriched['metadata'] = metadata_by_id[chunk_id]
        file.write(json.dumps(enriched, ensure_ascii=False) + '\n')
    file.flush()
    os.fsync(file.fileno())

os.replace(metadata_tmp, METADATA_PATH)
os.replace(enriched_tmp, ENRICHED_PATH)

print('Đã lưu metadata:', METADATA_PATH)
print('Đã lưu enriched chunks:', ENRICHED_PATH)
print('Số records:', len(ordered_metadata))


Final sanitize v4:   0%|          | 0/58603 [00:00<?, ?it/s]

Đã lưu metadata: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata/vn_history_rag_chunk_metadata.jsonl
Đã lưu enriched chunks: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata/vn_history_rag_chunks_enriched.jsonl
Số records: 58603


In [12]:
# Cell 11 — Báo cáo coverage và review thủ công

rows = []
for chunk in chunks:
    chunk_id = str(chunk['chunk_id'])
    metadata = metadata_by_id[chunk_id]

    rejected_count = sum(
        len(value) for value in metadata.get('rejected_items', {}).values()
    )
    entity_count = sum(
        len(metadata.get(field, []))
        for field in ['people', 'events', 'locations', 'documents', 'periods', 'dynasties']
    )

    needs_review = (
        metadata.get('extraction_status') != 'ok'
        or 'many_rejected_items' in metadata.get('quality_flags', [])
        or 'metadata_empty' in metadata.get('quality_flags', [])
    )

    rows.append({
        'chunk_id': chunk_id,
        'title': clean_text(chunk.get('title', '')),
        'source': clean_text(chunk.get('source', '')),
        'extraction_status': metadata.get('extraction_status', ''),
        'years_count': len(metadata.get('years', [])),
        'people_count': len(metadata.get('people', [])),
        'events_count': len(metadata.get('events', [])),
        'locations_count': len(metadata.get('locations', [])),
        'documents_count': len(metadata.get('documents', [])),
        'periods_count': len(metadata.get('periods', [])),
        'dynasties_count': len(metadata.get('dynasties', [])),
        'topics_count': len(metadata.get('topics', [])),
        'facets_count': len(metadata.get('content_facets', [])),
        'entity_count': entity_count,
        'rejected_count': rejected_count,
        'quality_flags': '|'.join(metadata.get('quality_flags', [])),
        'needs_review': needs_review,
    })

report_df = pd.DataFrame(rows)
report_df.to_csv(REPORT_PATH, index=False, encoding='utf-8-sig')

review_df = report_df[report_df['needs_review']].copy()
review_df.to_csv(REVIEW_PATH, index=False, encoding='utf-8-sig')

coverage = {}
for field in [
    'years_count', 'people_count', 'events_count', 'locations_count',
    'documents_count', 'periods_count', 'dynasties_count', 'topics_count',
    'facets_count',
]:
    coverage[field] = float((report_df[field] > 0).mean())

print('Coverage:')
for key, value in coverage.items():
    print(f'- {key}: {value:.1%}')

print(f'Chunks cần review: {len(review_df):,}/{len(report_df):,}')
print('Report:', REPORT_PATH)
print('Review list:', REVIEW_PATH)
display(report_df.head(10))


Coverage:
- years_count: 87.4%
- people_count: 40.4%
- events_count: 12.1%
- locations_count: 67.0%
- documents_count: 3.2%
- periods_count: 5.7%
- dynasties_count: 34.1%
- topics_count: 91.5%
- facets_count: 29.3%
Chunks cần review: 37,158/58,603
Report: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata/vn_history_rag_chunk_metadata_report.csv
Review list: /content/drive/MyDrive/vn_history_model_backups/rag_corpus_vn_history/metadata/vn_history_rag_chunk_metadata_manual_review.csv


,chunk_id,title,source,extraction_status,years_count,people_count,events_count,locations_count,documents_count,periods_count,dynasties_count,topics_count,facets_count,entity_count,rejected_count,quality_flags,needs_review
0,hf_wikipedia_thành_phố_hồ_chí_minh_0000_2da892...,Thành phố Hồ Chí Minh,DataStudio/Viet-wikipedia,ok,7,0,0,7,0,0,1,4,1,8,3,,False
1,hf_wikipedia_thành_phố_hồ_chí_minh_0001_ed1d52...,Thành phố Hồ Chí Minh,DataStudio/Viet-wikipedia,ok,15,0,0,3,0,0,4,4,0,7,6,many_rejected_items,True
2,hf_wikipedia_thành_phố_hồ_chí_minh_0002_4637b9...,Thành phố Hồ Chí Minh,DataStudio/Viet-wikipedia,ok,4,0,0,6,0,0,5,4,0,11,4,many_rejected_items,True
3,hf_wikipedia_thành_phố_hồ_chí_minh_0003_f0b3fd...,Thành phố Hồ Chí Minh,DataStudio/Viet-wikipedia,ok,12,0,0,8,0,0,3,4,0,11,9,many_rejected_items,True
4,hf_wikipedia_thành_phố_hồ_chí_minh_0004_2d5d49...,Thành phố Hồ Chí Minh,DataStudio/Viet-wikipedia,ok,11,3,0,5,0,0,2,4,0,10,15,many_rejected_items,True
5,hf_wikipedia_thành_phố_hồ_chí_minh_0005_c6c33d...,Thành phố Hồ Chí Minh,DataStudio/Viet-wikipedia,ok,3,0,0,0,0,1,2,4,0,3,8,many_rejected_items,True
6,hf_wikipedia_thành_phố_hồ_chí_minh_0006_3c6be5...,Thành phố Hồ Chí Minh,DataStudio/Viet-wikipedia,ok,3,0,0,7,0,0,1,3,0,8,7,many_rejected_items,True
7,hf_wikipedia_thành_phố_hồ_chí_minh_0007_5cda93...,Thành phố Hồ Chí Minh,DataStudio/Viet-wikipedia,ok,7,0,0,6,0,0,2,4,0,8,9,many_rejected_items,True
8,hf_wikipedia_thành_phố_hồ_chí_minh_0008_d771ad...,Thành phố Hồ Chí Minh,DataStudio/Viet-wikipedia,ok,4,0,0,1,0,0,1,4,0,2,5,many_rejected_items,True
9,hf_wikipedia_thành_phố_hồ_chí_minh_0009_546c86...,Thành phố Hồ Chí Minh,DataStudio/Viet-wikipedia,ok,3,0,0,1,0,0,1,2,1,2,1,,False


In [13]:
# Cell 12 — QC toàn bộ output: cấu trúc + semantic validators

metadata_records = read_jsonl(METADATA_PATH)
enriched_records = read_jsonl(ENRICHED_PATH)

corpus_ids = [str(chunk['chunk_id']) for chunk in chunks]
metadata_ids = [str(record.get('chunk_id', '')) for record in metadata_records]
enriched_ids = [str(record.get('chunk_id', '')) for record in enriched_records]

structural_errors = []
semantic_errors = []
join_errors = []

if len(metadata_records) != len(chunks):
    structural_errors.append(f'Metadata count {len(metadata_records)} != corpus count {len(chunks)}')
if len(enriched_records) != len(chunks):
    structural_errors.append(f'Enriched count {len(enriched_records)} != corpus count {len(chunks)}')
if metadata_ids != corpus_ids:
    structural_errors.append('Metadata chunk_id/order không khớp corpus.')
if enriched_ids != corpus_ids:
    structural_errors.append('Enriched chunk_id/order không khớp corpus.')
if len(set(metadata_ids)) != len(metadata_ids):
    structural_errors.append('Metadata có chunk_id trùng.')

field_error_counts = {field: 0 for field in ['years'] + SURFACE_FIELDS + ['dynasties']}
field_error_samples = {field: [] for field in field_error_counts}

for chunk, metadata, enriched in zip(chunks, metadata_records, enriched_records):
    chunk_id = str(chunk['chunk_id'])
    body = f"{clean_text(chunk.get('title', ''))}\n{clean_text(chunk.get('text', ''))}"

    if metadata.get('metadata_version') != METADATA_VERSION:
        structural_errors.append(f'{chunk_id}: sai metadata_version')
    if metadata.get('validator_signature') != VALIDATOR_SIGNATURE:
        structural_errors.append(f'{chunk_id}: sai validator_signature')
    if str(enriched.get('metadata', {}).get('chunk_id', '')) != chunk_id:
        join_errors.append(chunk_id)

    bad_years = [y for y in metadata.get('years', []) if not year_context_supported(y, body)]
    if bad_years:
        field_error_counts['years'] += 1
        field_error_samples['years'].append({'chunk_id': chunk_id, 'items': bad_years})

    for field in SURFACE_FIELDS:
        bad_items = [item for item in metadata.get(field, []) if not field_supported(field, item, body)]
        if bad_items:
            field_error_counts[field] += 1
            field_error_samples[field].append({'chunk_id': chunk_id, 'items': bad_items})

    expected_dynasties = extract_dynasties(body, max_items=8)
    if metadata.get('dynasties', []) != expected_dynasties:
        field_error_counts['dynasties'] += 1
        field_error_samples['dynasties'].append({
            'chunk_id': chunk_id,
            'actual': metadata.get('dynasties', []),
            'expected': expected_dynasties,
        })

if join_errors:
    structural_errors.append(f'Có {len(join_errors)} lỗi join metadata trong enriched file.')

for field, count in field_error_counts.items():
    if count:
        semantic_errors.append(f'{field}: {count} chunk không vượt validator cuối')

qc = {
    'metadata_version': METADATA_VERSION,
    'validator_signature': VALIDATOR_SIGNATURE,
    'corpus_count': len(chunks),
    'metadata_count': len(metadata_records),
    'enriched_count': len(enriched_records),
    'duplicate_metadata_ids': len(metadata_ids) - len(set(metadata_ids)),
    'join_error_count': len(join_errors),
    'semantic_error_counts': field_error_counts,
    'rules_fallback_count': int((report_df['extraction_status'] != 'ok').sum()),
    'review_count': int(report_df['needs_review'].sum()),
    'coverage': coverage,
    'structural_errors': structural_errors[:100],
    'semantic_errors': semantic_errors[:100],
    'ok': not structural_errors and not semantic_errors,
    'samples': {
        field: samples[:20]
        for field, samples in field_error_samples.items()
    },
}

with QC_PATH.open('w', encoding='utf-8') as file:
    json.dump(qc, file, ensure_ascii=False, indent=2)

print(json.dumps(qc, ensure_ascii=False, indent=2))
print('QC file:', QC_PATH)

if structural_errors or semantic_errors:
    raise RuntimeError('Output chưa vượt QC v4; xem QC JSON.')


{
  "metadata_version": "v4.0-final-people-location-strict",
  "validator_signature": "phase8_v4_people_location_20260731_a",
  "corpus_count": 58603,
  "metadata_count": 58603,
  "enriched_count": 58603,
  "duplicate_metadata_ids": 0,
  "join_error_count": 0,
  "semantic_error_counts": {
    "years": 0,
    "people": 0,
    "events": 0,
    "locations": 0,
    "documents": 0,
    "periods": 0,
    "dynasties": 0
  },
  "rules_fallback_count": 0,
  "review_count": 37158,
  "coverage": {
    "years_count": 0.8744603518591199,
    "people_count": 0.4040748767127963,
    "events_count": 0.12117127109533642,
    "locations_count": 0.6702045970342815,
    "documents_count": 0.03247274030339744,
    "periods_count": 0.057232564885756704,
    "dynasties_count": 0.34146716038428065,
    "topics_count": 0.9152944388512534,
    "facets_count": 0.2926812620514308
  },
  "structural_errors": [],
  "semantic_errors": [],
  "ok": true,
  "samples": {
    "years": [],
    "people": [],
    "events": 

In [14]:
# Cell 13 — Ví dụ join bằng chunk_id

metadata_by_chunk_id = {
    record['chunk_id']: record
    for record in metadata_records
}

sample_chunk = chunks[0]
sample_id = str(sample_chunk['chunk_id'])

print('CHUNK GỐC')
print(json.dumps({
    'chunk_id': sample_id,
    'title': sample_chunk.get('title', ''),
    'text_preview': clean_text(sample_chunk.get('text', ''))[:500],
}, ensure_ascii=False, indent=2))

print('\nMETADATA JOIN QUA chunk_id')
print(json.dumps(
    metadata_by_chunk_id[sample_id],
    ensure_ascii=False,
    indent=2,
))


CHUNK GỐC
{
  "chunk_id": "hf_wikipedia_thành_phố_hồ_chí_minh_0000_2da892be6bd2",
  "title": "Thành phố Hồ Chí Minh",
  "text_preview": "Thành phố Hồ Chí Minh (viết tắt TP.HCM), còn được gọi là Sài Gòn, là thành phố lớn nhất Việt Nam và là một siêu đô thị trong tương lai gần. Đây còn là trung tâm kinh tế, giải trí, một trong hai trung tâm văn hóa và giáo dục quan trọng tại Việt Nam. Thành phố Hồ Chí Minh là thành phố trực thuộc trung ương thuộc loại đô thị đặc biệt của Việt Nam. Nằm trong vùng chuyển tiếp giữa Đông Nam Bộ và Tây Nam Bộ, thành phố này hiện có 16 quận, 1 thành phố và 5 huyện, tổng diện tích 2.095 km2 (809 dặm vuông"
}

METADATA JOIN QUA chunk_id
{
  "chunk_id": "hf_wikipedia_thành_phố_hồ_chí_minh_0000_2da892be6bd2",
  "metadata_version": "v4.0-final-people-location-strict",
  "validator_signature": "phase8_v4_people_location_20260731_a",
  "extraction_method": "hybrid",
  "extraction_status": "ok",
  "source_sha1": "31c83208fa7f1a5e2924876fe080037375858f1c",
  "years": [

In [15]:
# ============================================================
# DELAY → DISCONNECT COLAB RUNTIME
# ============================================================

import time

DELAY_MINUTES = 5

print(f"Phase 8 đã hoàn tất.")
print(f"Sẽ ngắt runtime sau {DELAY_MINUTES} phút...")

time.sleep(DELAY_MINUTES * 60)

print("Đang ngắt runtime...")

from google.colab import runtime
runtime.unassign()

Phase 8 đã hoàn tất.
Sẽ ngắt runtime sau 5 phút...
Đang ngắt runtime...


## Cách dùng trong retrieval

- Join metadata với corpus bằng `chunk_id`.
- Metadata chỉ dùng để **boost mềm** hoặc rerank; không nên lọc cứng ngay từ đầu.
- `title + text` vẫn là nội dung dùng để tạo embedding.
- Khi đưa context cho Qwen đã SFT, vẫn giữ format `[chunk_id] title\ntext` như Phase 6.
- File v3 ưu tiên precision, nên một số field có thể rỗng; điều đó an toàn hơn metadata sai.

- V4 ưu tiên precision: `people` và `locations` có thể thưa hơn, nhưng giảm nguy cơ boost sai thực thể.
